In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv




In [11]:
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # fast + cheap
    temperature=0.7
)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [12]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str



In [13]:
def create_outline(state:BlogState)->BlogState:
    #fetch title
    title = state['title']

    #call llm to create outline 
    prompt = f'Generate a detailed outline for a blog post on the topic - {title}'
    outline = model.invoke(prompt).content


    #update state with outline 
    state['outline']=outline 

    return state


In [6]:
def create_blog(state:BlogState)->BlogState:
    #fetch title and outline 
    title = state['title']
    outline = state['outline']

    #call llm to create blog 
    prompt = f'Create a detailed blog post on the topic - {title} with the following outline \n {outline}'
    content = model.invoke(prompt).content

    #update state with content 
    state['content']=content 

    return state


In [9]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

#edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [14]:
initial_state = {'title': 'Rise of AI'}
final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI', 'outline': 'Here\'s a detailed outline for a blog post on "The Rise of AI," designed to be engaging, informative, and thought-provoking.\n\n---\n\n## Blog Post Title Ideas:\n\n*   The Rise of AI: Understanding the Revolution That\'s Reshaping Our World\n*   Beyond the Hype: Decoding the Unstoppable Rise of Artificial Intelligence\n*   The Dawn of a New Era: Navigating the Rapid Ascent of AI\n*   From Sci-Fi to Reality: How AI is Taking Over (and What It Means for You)\n\n---\n\n## Detailed Blog Post Outline: The Rise of AI\n\n**I. Introduction: The AI Awakening**\n\n*   **A. Catchy Hook:** Start with a bold statement or a relatable scenario about AI\'s current omnipresence (e.g., "AI isn\'t just science fiction anymore; it\'s the invisible hand guiding your recommendations, powering your smart devices, and now, generating your content.")\n*   **B. Brief Definition of AI:** Acknowledge that AI is a broad term, but focus on the current understanding – machines des

In [15]:
print(final_state['outline'])


Here's a detailed outline for a blog post on "The Rise of AI," designed to be engaging, informative, and thought-provoking.

---

## Blog Post Title Ideas:

*   The Rise of AI: Understanding the Revolution That's Reshaping Our World
*   Beyond the Hype: Decoding the Unstoppable Rise of Artificial Intelligence
*   The Dawn of a New Era: Navigating the Rapid Ascent of AI
*   From Sci-Fi to Reality: How AI is Taking Over (and What It Means for You)

---

## Detailed Blog Post Outline: The Rise of AI

**I. Introduction: The AI Awakening**

*   **A. Catchy Hook:** Start with a bold statement or a relatable scenario about AI's current omnipresence (e.g., "AI isn't just science fiction anymore; it's the invisible hand guiding your recommendations, powering your smart devices, and now, generating your content.")
*   **B. Brief Definition of AI:** Acknowledge that AI is a broad term, but focus on the current understanding – machines designed to simulate human intelligence, learning, and problem

In [16]:
print(final_state['content'])

## The Rise of AI: Understanding the Revolution That's Reshaping Our World

AI isn't just science fiction anymore; it's the invisible hand guiding your recommendations, powering your smart devices, and now, generating your content. What was once the realm of futuristic novels and blockbuster movies has rapidly become an undeniable force woven into the fabric of our daily lives. At its core, Artificial Intelligence refers to machines designed to simulate human intelligence, learning, problem-solving, and decision-making. But what truly sets this current era apart is the unprecedented speed and scale of AI's ascent.

This isn't just another technological advancement; it's a fundamental shift poised to redefine industries, reshape societies, and challenge our very understanding of intelligence. This post will explore the key factors driving AI's rapid ascent, its transformative impact across industries and daily life, the immense promises it holds, and the critical challenges we must navi